# Análisis exploratorio de dimensiones
Este notebook carga los archivos `dim_*.parquet` desde `data/processedPedro` y realiza un análisis básico con `pandas` y `scikit-learn`.

Pasos:
1. Importar librerías.
2. Cargar cada dimensión con su nombre referencial.
3. Mostrar shape, columnas, primeras 10 filas y descripción básica.
4. Revisar valores nulos y codificación simple de columnas categóricas.


In [1]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data" / "processedPedro"
DIM_FILES = {
    "dim_date": "dim_date.parquet",
    "dim_device": "dim_device.parquet",
    "dim_episode": "dim_episode.parquet",
    "dim_location": "dim_location.parquet",
    "dim_location_enriched": "dim_location_enriched.parquet",
    "dim_track": "dim_track.parquet",
}

print("Directorio de datos:", DATA_DIR)
if DATA_DIR.exists():
    print("Archivos disponibles:")
    for path in sorted(DATA_DIR.glob("dim_*.parquet")):
        print(" -", path.name)
else:
    print("ERROR: data/processedPedro no existe o no es accesible.")


Directorio de datos: C:\Users\gal0i\Proyectos\portafolio-spotify\data\processedPedro
Archivos disponibles:
 - dim_date.parquet
 - dim_device.parquet
 - dim_episode.parquet
 - dim_location.parquet
 - dim_location_enriched.parquet
 - dim_track.parquet


In [7]:

def load_dimensions(data_dir: Path):
    dims = {}
    for key, filename in DIM_FILES.items():
        path = data_dir / filename
        if path.exists():
            dims[key] = pd.read_parquet(path)
        else:
            print(f"Warning: falta {filename}")
    return dims


def analyze_dimension(name: str, df: pd.DataFrame):
    print("\n" + "=" * 80)
    print(f"Dimensión: {name}")
    print("=" * 80)
    print(f"Shape: {df.shape[0]} filas x {df.shape[1]} columnas")
    print("Columns:", list(df.columns))
    print("\nPrimeras 10 filas:")
    display(df.head(10))
    print("\nValores nulos por columna:")
    print(df.isna().sum())
    print("\nDescripción estadística:")
    try:
        display(df.describe(include='all', datetime_is_numeric=True).transpose())
    except TypeError:
        display(df.describe(include='all').transpose())

    categorical = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    if categorical:
        print("\nColumnas categóricas:", categorical)
        for col in categorical:
            values = df[col].fillna("[NA]").astype(str)
            print(f"- {col}: {values.nunique(dropna=False)} valores únicos")
            print("  Ejemplos:", values.drop_duplicates().head(5).tolist())
            if values.nunique(dropna=False) <= 20:
                encoded = LabelEncoder().fit_transform(values)
                print("  Codificación LabelEncoder (primeros 5):", encoded[:5].tolist())
    else:
        print("\nNo se detectaron columnas categóricas.")


dimensions = load_dimensions(DATA_DIR)
for name, df in dimensions.items():
    analyze_dimension(name, df)



Dimensión: dim_date
Shape: 14843 filas x 6 columnas
Columns: ['date_id', 'year', 'month', 'day', 'weekday', 'hour']

Primeras 10 filas:


,date_id,year,month,day,weekday,hour
0,2017020920,2017,2,9,Thursday,20
1,2017020921,2017,2,9,Thursday,21
2,2017020922,2017,2,9,Thursday,22
3,2017020923,2017,2,9,Thursday,23
4,2017021000,2017,2,10,Friday,0
5,2017021001,2017,2,10,Friday,1
6,2017021003,2017,2,10,Friday,3
7,2017021004,2017,2,10,Friday,4
8,2017021019,2017,2,10,Friday,19
9,2017021023,2017,2,10,Friday,23



Valores nulos por columna:
date_id    0
year       0
month      0
day        0
weekday    0
hour       0
dtype: int64

Descripción estadística:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date_id,14843,14843,2017020920,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,14843.0,NaN,NaN,NaN,2022.69137,1.795223,2017.0,2022.0,2023.0,2024.0,2025.0
month,14843.0,NaN,NaN,NaN,6.510746,3.244402,1.0,4.0,6.0,9.0,12.0
day,14843.0,NaN,NaN,NaN,15.792091,8.717707,1.0,8.0,16.0,23.0,31.0
weekday,14843,7,Tuesday,2396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hour,14843.0,NaN,NaN,NaN,12.296975,8.054938,0.0,4.0,15.0,20.0,23.0



Columnas categóricas: ['date_id', 'weekday']
- date_id: 14843 valores únicos
  Ejemplos: ['2017020920', '2017020921', '2017020922', '2017020923', '2017021000']
- weekday: 7 valores únicos
  Ejemplos: ['Thursday', 'Friday', 'Saturday', 'Sunday', 'Monday']
  Codificación LabelEncoder (primeros 5): [4, 4, 4, 4, 0]

Dimensión: dim_device
Shape: 17 filas x 3 columnas
Columns: ['platform', 'device_type', 'device_id']

Primeras 10 filas:


...


Columnas categóricas: ['ip_addr', 'conn_country', 'city', 'country', 'region', 'isp']
- ip_addr: 2242 valores únicos
  Ejemplos: ['0.0.0.0', '179.4.182.122', '179.4.182.x', '179.4.190.140', '179.4.180.229']
- conn_country: 2 valores únicos
  Ejemplos: ['XX', 'CL']
  Codificación LabelEncoder (primeros 5): [1, 0, 0, 0, 0]
- city: 10 valores únicos
  Ejemplos: ['Unknown', 'Los Ángeles', 'Arica', 'Curicó', 'Punta Arenas']
  Codificación LabelEncoder (primeros 5): [7, 3, 7, 3, 3]
- country: 2 valores únicos
  Ejemplos: ['[NA]', 'Chile']
  Codificación LabelEncoder (primeros 5): [1, 0, 1, 0, 0]
- region: 8 valores únicos
  Ejemplos: ['Unknown', 'Región del Biobío', 'Arica y Parinacota Region', 'Maule Region', 'Region of Magallanes']
  Codificación LabelEncoder (primeros 5): [7, 5, 7, 5, 5]
- isp: 8 valores únicos
  Ejemplos: ['Unknown', 'Telmex Servicios Empresariales S.A.', 'Pacifico Cable SPA.', 'VTR BANDA ANCHA S.A.', 'WOM S.A.']
  Codificación LabelEncoder (primeros 5): [5, 3, 5, 3, 3]

,spotify_track_uri,track_name,artist_name,album_name,track_id
0,Unknown,Non-Music Content,N/A,N/A,-1
1,spotify:track:57Xjny5yNzAcsxnusKmAfA,Reptilia,The Strokes,Room On Fire,0
2,spotify:track:4kTd0TND65MUY4BlcmJ2cM,Why'd You Only Call Me When You're High?,Arctic Monkeys,AM,1
3,spotify:track:05FVS9qqWq2A82fYCtblkt,Who The Fuck Are Arctic Monkeys?,Arctic Monkeys,Who The F*** Are Arctic Monkeys?,2
4,spotify:track:2UzMpPKPhbcC8RbsmuURAZ,Do I Wanna Know?,Arctic Monkeys,AM,3
5,spotify:track:7kzKAuUzOITUauHAhoMoxA,Last Nite,The Strokes,Is This It,4
6,spotify:track:0GRvnYqlnh7ttq9NS6s3Ta,Desecration Smile,Red Hot Chili Peppers,Stadium Arcadium,5
7,spotify:track:2XTkpF9T2PKvcLgamGJGx1,Goodbye Angels,Red Hot Chili Peppers,The Getaway,6
8,spotify:track:5qqabIl2vWzo9ApSC317sa,Wonderwall - Remastered,Oasis,(What's The Story) Morning Glory?,7
9,spotify:track:0uXNXawMM10jqdzMSIbO4l,Songbird,Oasis,Heathen Chemistry,8



Valores nulos por columna:
spotify_track_uri    0
track_name           0
artist_name          0
album_name           0
track_id             0
dtype: int64

Descripción estadística:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
spotify_track_uri,15438,15438,Unknown,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_name,15438,13755,Intro,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
artist_name,15438,2749,Red Hot Chili Peppers,238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
album_name,15438,5676,Mellon Collie And The Infinite Sadness,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_id,15438.0,NaN,NaN,NaN,7717.5,4456.711063,-1.0,3858.25,7717.5,11576.75,15436.0



Columnas categóricas: ['spotify_track_uri', 'track_name', 'artist_name', 'album_name']
- spotify_track_uri: 15438 valores únicos
  Ejemplos: ['Unknown', 'spotify:track:57Xjny5yNzAcsxnusKmAfA', 'spotify:track:4kTd0TND65MUY4BlcmJ2cM', 'spotify:track:05FVS9qqWq2A82fYCtblkt', 'spotify:track:2UzMpPKPhbcC8RbsmuURAZ']
- track_name: 13755 valores únicos
  Ejemplos: ['Non-Music Content', 'Reptilia', "Why'd You Only Call Me When You're High?", 'Who The Fuck Are Arctic Monkeys?', 'Do I Wanna Know?']
- artist_name: 2749 valores únicos
  Ejemplos: ['N/A', 'The Strokes', 'Arctic Monkeys', 'Red Hot Chili Peppers', 'Oasis']
- album_name: 5676 valores únicos
  Ejemplos: ['N/A', 'Room On Fire', 'AM', 'Who The F*** Are Arctic Monkeys?', 'Is This It']
